# Out-of-bag (OOB) evaluation
A random forest on the heart disease data with `oob_score=True`, the OOB score rebuilt by hand, and the OOB score
of a regressor.

In [ ]:
import warnings

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import cross_val_score, train_test_split

## 1. The data: 303 patients, 242 for training, 61 for testing

In [ ]:
df = pd.read_csv("data/heart.csv")
X = df.iloc[:, :-1]
y = df.iloc[:, -1]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

## 2. The OOB score

In [ ]:
rf = RandomForestClassifier(oob_score=True, random_state=42)   # oob_score=True must be set before fit
rf.fit(X_train, y_train)
print("OOB score:      ", round(rf.oob_score_, 3))
print("test accuracy:  ", round(rf.score(X_test, y_test), 3))
cv = cross_val_score(RandomForestClassifier(random_state=42), X_train, y_train, cv=5).mean()
print("5-fold CV score:", round(cv, 3), "(trains 5 more forests)")

## 3. Which rows each tree missed
`estimators_samples_[t]` holds the row positions tree t was trained on (with repeats).

In [ ]:
n_trees, n_rows = len(rf.estimators_), len(X_train)
in_bag = np.zeros((n_trees, n_rows), dtype=bool)
for t, rows in enumerate(rf.estimators_samples_):
    in_bag[t, rows] = True
oob = ~in_bag                                   # True where tree t never saw row i

print("share of rows that are OOB, first 5 trees:", oob.mean(axis=1)[:5].round(3))
print("average share of OOB rows per tree:", oob.mean().round(3))
print("each row is OOB for this many trees: min", oob.sum(axis=0).min(), " mean", oob.sum(axis=0).mean().round(1),
      " max", oob.sum(axis=0).max())
print("rows that every tree saw:", int((oob.sum(axis=0) == 0).sum()))

## 4. The OOB score by hand
For each row, add up the class probabilities of the trees that never saw it, pick the larger class, compare with
the truth.

In [ ]:
X_arr, y_arr = X_train.values, y_train.values
proba = np.zeros((n_rows, 2))
for t, tree in enumerate(rf.estimators_):
    proba[oob[t]] += tree.predict_proba(X_arr[oob[t]])      # only this tree's OOB rows
oob_pred = proba.argmax(axis=1)
print("OOB accuracy by hand:", round((oob_pred == y_arr).mean(), 3))
print("matches oob_decision_function_:", np.allclose(proba / proba.sum(axis=1, keepdims=True),
                                                     rf.oob_decision_function_))
rf.oob_decision_function_[:3]

## 5. Too few trees
With 5 trees, some rows are in every tree's sample and get no OOB prediction; scikit-learn warns.

In [ ]:
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    small = RandomForestClassifier(n_estimators=5, oob_score=True, random_state=42).fit(X_train, y_train)
print("OOB score with 5 trees:", round(small.oob_score_, 3))
print("rows with no OOB tree:", int((small.oob_decision_function_.sum(axis=1) == 0).sum()), "of", n_rows)
print([str(m.message)[:110] for m in w])

In [ ]:
for n in [5, 10, 20, 50, 100, 200, 500]:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        m = RandomForestClassifier(n_estimators=n, oob_score=True, random_state=42, n_jobs=-1).fit(X_train, y_train)
    print(f"{n:4d} trees: OOB {m.oob_score_:.3f}   test {m.score(X_test, y_test):.3f}")

## 6. OOB needs bootstrap=True

In [ ]:
try:
    RandomForestClassifier(bootstrap=False, oob_score=True).fit(X_train, y_train)
except ValueError as e:
    print("ValueError:", e)

## 7. A regressor: the OOB score is R²
California housing, a sample of 5,000 districts (`fetch_california_housing` downloads it once).

In [ ]:
Xh, yh = fetch_california_housing(return_X_y=True)
idx = np.random.default_rng(0).choice(len(Xh), 5000, replace=False)
Xh_train, Xh_test, yh_train, yh_test = train_test_split(Xh[idx], yh[idx], test_size=0.2, random_state=42)
rfr = RandomForestRegressor(oob_score=True, random_state=42, n_jobs=-1).fit(Xh_train, yh_train)
print("OOB R2: ", round(rfr.oob_score_, 3))
print("test R2:", round(rfr.score(Xh_test, yh_test), 3))
print("first OOB predictions:", rfr.oob_prediction_[:3].round(2), " true:", yh_train[:3])